This tutorial covers the end-to-end workflow for the hackathon: **Installation → Calibration → Data Collection → Inference**.

This course is built on [**LeRobot**](https://github.com/huggingface/lerobot), an open-source robotics library developed by [**Hugging Face**](https://huggingface.co/) 🤗. We extend our sincere thanks to the Hugging Face team for making accessible, high-quality robotics research tools available to everyone.

📖 **Documentation:** [https://huggingface.co/docs/lerobot](https://huggingface.co/docs/lerobot)

## Environment & Installation

**Crucial Requirement:** You must use **Python 3.10**. The robot software is sensitive to version mismatches.

### Install System Dependencies

**For Mac (Apple Silicon/Intel):**
You must use **FFmpeg version 7**. Version 8 (current default) is known to cause issues.

```bash
# 1. Install Python 3.10
brew install python@3.10

# 2. Handle FFmpeg (Downgrade to v7)
ffmpeg -version  # Check current version
brew uninstall ffmpeg
brew install ffmpeg@7
brew link ffmpeg@7

# 3. Add to path (Run this, then restart terminal)
export PATH="/opt/homebrew/opt/ffmpeg@7/bin:$PATH"
```

**For Windows:**
Ensure Python 3.10 is installed and added to your PATH.

### Python Environment Setup

::: {.panel-tabset}

## venv (Recommended)

**1. Clone the repo and navigate to the folder:**

```bash
git clone https://github.com/huggingface/lerobot.git
cd lerobot
```

**2. Create and Activate Virtual Environment:**

**Mac/Linux:**
```bash
# Create venv
python3.10 -m venv .venv

# Activate
source .venv/bin/activate
```

**Windows:**
```powershell
# Create venv
python3.10 -m venv .venv

# Activate
.venv\Scripts\activate
```

**3. Install LeRobot:**

```bash
pip install --upgrade pip
pip install -e .
pip install -e ".[feetech]"
```

**Daily Usage:**
```bash
cd /path/to/lerobot
source .venv/bin/activate
```

## Miniforge (Conda)

**1. Install Miniforge:**

```bash
cd ~
curl -L "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Darwin-arm64.sh" -o Miniforge3-installer.sh
bash Miniforge3-installer.sh -b -p ~/miniforge3
rm Miniforge3-installer.sh
```

**2. Initialize Conda:**

```bash
~/miniforge3/bin/conda init zsh
source ~/miniforge3/etc/profile.d/conda.sh
```

**3. Create Environment:**

```bash
conda create -y -n lerobot python=3.10
conda activate lerobot
```

**4. Install FFmpeg:**

```bash
conda install -y ffmpeg -c conda-forge
```

**5. Clone and Install LeRobot:**

```bash
git clone https://github.com/huggingface/lerobot.git
cd lerobot
pip install -e .
pip install -e ".[feetech]"
```

**Daily Usage:**
```bash
conda activate lerobot
```

**Verify Installation:**
```bash
python --version                    # Should be 3.10.x
pip show lerobot | grep Version     # Should show lerobot version
ffmpeg -encoders 2>&1 | grep svtav1 # Should show libsvtav1
```

:::

## 2. Hardware Setup & Assembly

The follower arm uses 6x STS3215 (12V) motors with 1/345 gearing. The leader, however, uses three differently geared motors to make sure it can both sustain its own weight and it can be moved without requiring much force.

| Leader-Arm Axis | Motor | Gear Ratio |
|---|---|---|
| Base / Shoulder Pan | 1 | 1 / 191 |
| Shoulder Lift | 2 | 1 / 345 |
| Elbow Flex | 3 | 1 / 191 |
| Wrist Flex | 4 | 1 / 147 |
| Wrist Roll | 5 | 1 / 147 |
| Gripper | 6 | 1 / 147 |

Please mark them from 1 to 6 for both the leader and the follower.

### Motor Specifications

In this section, we explain how to assemble the SO-101 robot arm.

### Step 1: Identify Ports

Connect the 2 USB MotorBus to your computer. They are the motherboards that are used to control the arms. Make also sure to connect the alimentation cable and connect the black to the - and the red to the +.

Each Bus will have a different port of the computer linked to it and we need to find them.

```bash
lerobot-find-port
```

Example output:
```bash
Finding all available ports for the MotorBus.
['/dev/tty.usbmodem575E0032081', '/dev/tty.usbmodem575E0031751']
Remove the USB cable from your MotorsBus and press Enter when done.

[...Disconnect corresponding leader or follower arm and press Enter...]

The port of this MotorsBus is /dev/tty.usbmodem575E0032081
Reconnect the USB cable.
```

**Copy the paths** detected (e.g., `/dev/tty.usbmodem...`). You will need one for the Leader and one for the Follower.

### Step 2: Motor Configuration

Each motor is identified by a unique id on the bus. When brand new, motors usually come with a default id of 1. For the communication to work properly between the motors and the controller, we first need to set a unique, different id to each motor. Additionally, the speed at which data is transmitted on the bus is determined by the baudrate. In order to talk to each other, the controller and all the motors need to be configured with the same baudrate.

We first need to connect to each motor individually with the controller in order to set these. Since we will write these parameters in the non-volatile section of the motors' internal memory (EEPROM), we'll only need to do this once.

The video below shows the sequence of steps for setting the motor ids:

{{< video assets/robot_assembly/setup_motors_so101_2.mp4 width="600" >}}

Connect the usb cable from your computer and the power supply to the arm's controller board.

**Follower Arm Configuration:**

```bash
lerobot-setup-motors \
    --robot.type=so101_follower \
    --robot.port=/dev/tty.usbmodem585A0076841  # <- PASTE YOUR PORT HERE
```

You should see the following instruction:
```
Connect the controller board to the 'gripper' motor only and press enter.
```

As instructed, plug the gripper's motor. Make sure it's the only motor connected to the board, and that the motor itself is not yet daisy-chained to any other motor. As you press [Enter], the script will automatically set the id and baudrate for that motor.

You should then see:
```
'gripper' motor id set to 6
```

Followed by the next instruction for each motor. Repeat the operation for each motor as instructed.

**Leader Arm Configuration:**

```bash
lerobot-setup-motors \
    --teleop.type=so101_leader \
    --teleop.port=/dev/tty.usbmodem575E0031751  # <- PASTE YOUR PORT HERE
```

When done, the motors are ready to be used. You can now plug the 3-pin cable from each motor to the next one, and the cable from the first motor (the 'shoulder pan' with id=1) to the controller board.

### Step 3: Build the Robot

Remove all support material from the 3D-printed parts. The easiest way to do this is using a small screwdriver to get underneath the support material.

**It is advisable to install one 3-pin cable in the motor after placing them before continuing assembly.**

---

**Joint 1**

{{< video assets/robot_assembly/Joint1_v2.mp4 width="600" >}}

- Place the first motor into the base
- Fasten the motor with 4 M2x6mm screws (smallest screws). Two from the top and two from the bottom
- Slide over the first motor holder and fasten it using two M2x6mm screws (one on each side)
- Install both motor horns, securing the top horn with a M3x6mm screw
- Attach the shoulder part
- Tighten the shoulder part with 4 M3x6mm screws on top and 4 M3x6mm screws on the bottom
- Add the shoulder motor holder

---

**Joint 2**

{{< video assets/robot_assembly/Joint2_v2.mp4 width="600" >}}

- Slide the second motor in from the top
- Fasten the second motor with 4 M2x6mm screws
- Attach both motor horns to motor 2, again use the M3x6mm horn screw
- Attach the upper arm with 4 M3x6mm screws on each side

---

**Joint 3**

{{< video assets/robot_assembly/Joint3_v2.mp4 width="600" >}}

- Insert motor 3 and fasten using 4 M2x6mm screws
- Attach both motor horns to motor 3 and secure one again with a M3x6mm horn screw
- Connect the forearm to motor 3 using 4 M3x6mm screws on each side

---

**Joint 4**

{{< video assets/robot_assembly/Joint4_v2.mp4 width="600" >}}

- Slide over motor holder 4
- Slide in motor 4
- Fasten motor 4 with 4 M2x6mm screws and attach its motor horns, use a M3x6mm horn screw

---

**Joint 5**

{{< video assets/robot_assembly/Joint5_v2.mp4 width="600" >}}

- Insert motor 5 into the wrist holder and secure it with 2 M2x6mm front screws
- Install only one motor horn on the wrist motor and secure it with a M3x6mm horn screw
- Secure the wrist to motor 4 using 4 M3x6mm screws on both sides

---

**Gripper / Handle**

{{< video assets/robot_assembly/Gripper_v2.mp4 width="600" >}}

**Follower:**

- Attach the gripper to motor 5, attach it to the motor horn on the wrist using 4 M3x6mm screws
- Insert the gripper motor and secure it with 2 M2x6mm screws on each side
- Attach the motor horns and again use a M3x6mm horn screw
- Install the gripper claw and secure it with 4 M3x6mm screws on both sides

**Leader:**

- Mount the leader holder onto the wrist and secure it with 4 M3x6mm screws
- Attach the handle to motor 5 using 1 M2x6mm screw
- Insert the gripper motor, secure it with 2 M2x6mm screws on each side, attach a motor horn using a M3x6mm horn screw
- Attach the follower trigger with 4 M3x6mm screws

## Calibration

Once built, you must calibrate the arms so the software knows the physical range of motion. The calibration process is very important because it allows a neural network trained on one robot to work on another.

The video below shows how to perform the calibration. First you need to move the robot to the position where all joints are in the middle of their ranges. Then after pressing enter you have to move each joint through its full range of motion.

{{< video assets/robot_assembly/calibrate_so101_2.mp4 width="600" >}}

**Calibrate Follower:**

```bash
lerobot-calibrate \
    --robot.type=so101_follower \
    --robot.port=/dev/tty.usbmodem5AAF2628581  # <- PASTE YOUR PORT HERE \
    --robot.id=my_awesome_follower_arm
```

**Calibrate Leader:**

```bash
lerobot-calibrate \
    --teleop.type=so101_leader \
    --teleop.port=/dev/tty.usbmodem5AAF2632261  # <- PASTE YOUR PORT HERE \
    --teleop.id=my_awesome_leader_arm
```

*Note: Use the unique IDs `my_awesome_follower_arm` and `my_awesome_leader_arm` consistently.*

## Teleoperation (Testing)

Before recording, ensure the Leader moves the Follower correctly.

**Test 1: Blind Teleoperation (No Camera)**

```bash
lerobot-teleoperate \
    --robot.type=so101_follower \
    --robot.port=/dev/tty.usbmodem5AAF2628581  # <- PASTE YOUR PORT HERE \
    --robot.id=my_awesome_follower_arm \
    --teleop.type=so101_leader \
    --teleop.port=/dev/tty.usbmodem5AAF2632261  # <- PASTE YOUR PORT HERE \
    --teleop.id=my_awesome_leader_arm
```

**Test 2: With Camera Visualization (1 camera)**

If you have a webcam connected (Index 0).

```bash
lerobot-teleoperate \
    --robot.type=so101_follower \
    --robot.port=/dev/tty.usbmodem5AAF2628581  # <- PASTE YOUR PORT HERE \
    --robot.id=my_awesome_follower_arm \
    --robot.cameras="{ webcam: {type: opencv, index_or_path: 0, width: 640, height: 480, fps: 20}}" \
    --teleop.type=so101_leader \
    --teleop.port=/dev/tty.usbmodem5AAF2632261  # <- PASTE YOUR PORT HERE \
    --teleop.id=my_awesome_leader_arm \
    --display_data=true
```

**Test 3: With 2 Cameras**

If you have 2 cameras (Index 0 and 1).

```bash
lerobot-teleoperate \
    --robot.type=so101_follower \
    --robot.port=/dev/tty.usbmodem5AAF2628581  # <- PASTE YOUR PORT HERE \
    --robot.id=my_awesome_follower_arm \
    --robot.cameras="{ webcam: {type: opencv, index_or_path: 0, width: 640, height: 480, fps: 20}, computer: {type: opencv, index_or_path: 1, width: 640, height: 480, fps: 20}}" \
    --teleop.type=so101_leader \
    --teleop.port=/dev/tty.usbmodem5AAF2632261  # <- PASTE YOUR PORT HERE \
    --teleop.id=my_awesome_leader_arm \
    --display_data=true
```

## Record Dataset

This is the most critical step. You will generate the data used to train the AI.

**1. Set Team Name Variable:**

```bash
TEAM="team_name"
```

**2. Start Recording:**

Run the following command. Note that `--dataset.push_to_hub=False` is set to keep data local for now.

```bash
lerobot-record \
    --robot.type=so101_follower \
    --robot.port=/dev/tty.usbmodem5AAF2628581  # <- PASTE YOUR PORT HERE \
    --robot.id=my_awesome_follower_arm \
    --robot.cameras="{ webcam: {type: opencv, index_or_path: 0, width: 640, height: 480, fps: 20}, computer: {type: opencv, index_or_path: 1, width: 640, height: 480, fps: 20}}" \
    --teleop.type=so101_leader \
    --teleop.port=/dev/tty.usbmodem5AAF2632261  # <- PASTE YOUR PORT HERE \
    --teleop.id=my_awesome_leader_arm \
    --display_data=true \
    --dataset.repo_id=${TEAM}/record-test \
    --dataset.num_episodes=30 \
    --dataset.single_task="Grab the black cube" \
    --dataset.push_to_hub=False \
    --dataset.episode_time_s=30 \
    --dataset.reset_time_s=20
```

**Resume Recording:**

If you need to add episodes to an existing batch, append:

```bash
--resume=true
```

**Replay a specific episode (Quality Check):**

```bash
lerobot-replay \
    --robot.type=so101_follower \
    --robot.port=/dev/tty.usbmodem5AAF2628581  # <- PASTE YOUR PORT HERE \
    --robot.id=my_awesome_follower_arm \
    --dataset.repo_id=${TEAM}/record-test \
    --dataset.episode=0
```

### Sharing Data for Training

The training will be handled externally. You need to export your dataset.

1. **Locate the dataset:**
   Go to: `~/.cache/huggingface/lerobot/{TEAM}`.
   **Mac tip:** Press `Cmd+Shift+G` in Finder and paste the path.

2. **Zip the folders:**
   Ensure the zip file contains these three folders:
   - `meta`
   - `data`
   - `videos`

3. **Upload:** Upload the zip file to the Team Google Drive.

## Training (Google Colab)

*Note: This step is performed by the Emerton Data team on Google Colab.*

For reference, the training command used is:

```bash
lerobot-train \
  --dataset.repo_id=${TEAM}/record-test \
  --policy.type=act \
  --output_dir=outputs/train/act_so101_test \
  --job_name=act_so101_test \
  --policy.device=mps \  # Use 'cuda' for NVIDIA, 'mps' for Mac M-chips
  --wandb.enable=false \
  --policy.repo_id=${TEAM}/my_policy
```

## Inference (Running the Model)

Once the Emerton team returns the trained model, follow these steps to run it on your robot.

**1. Place Model Files:**

Put the trained model into your `lerobot` directory:

- Path: `outputs/{TEAM}/`
- Must contain two folders inside: `pretrained_model` and `training_state`.

**2. Run Inference:**

We use `lerobot-record` for inference by passing the `--policy.path` argument.

```bash
lerobot-record \
  --robot.type=so101_follower \
  --robot.port=/dev/tty.usbmodem5AAF2628581  # <- PASTE YOUR PORT HERE \
  --robot.cameras="{ webcam: {type: opencv, index_or_path: 0, width: 640, height: 480, fps: 20}, computer: {type: opencv, index_or_path: 1, width: 640, height: 480, fps: 20}}" \
  --robot.id=my_awesome_follower_arm \
  --display_data=false \
  --dataset.repo_id=${TEAM}/eval_so101-2 \
  --dataset.single_task="Put lego brick into the transparent box" \
  --policy.path=outputs/${TEAM}/pretrained_model \
  --dataset.push_to_hub=False
```

## Command Reference

| Action | Command |
|--------|--------|
| Find ports | `lerobot-find-port` |
| Setup motors | `lerobot-setup-motors` |
| Calibrate | `lerobot-calibrate` |
| Teleoperate | `lerobot-teleoperate` |
| Record data | `lerobot-record` |
| Replay episode | `lerobot-replay` |
| Train model | `lerobot-train` |
| Check Python version | `python --version` |
| List installed packages | `pip list` |

## Acknowledgments

This practical course and hackathon would not be possible without the incredible work of the **Hugging Face** team on the **LeRobot** library. Special thanks to:

- [**Hugging Face**](https://huggingface.co/) 🤗 for developing and open-sourcing LeRobot
- The LeRobot contributors for creating accessible robotics research tools
- The open-source AI community for advancing robotics research

**Resources:**

- [LeRobot GitHub Repository](https://github.com/huggingface/lerobot)
- [LeRobot Documentation](https://huggingface.co/docs/lerobot)
- [Hugging Face Blog](https://huggingface.co/blog)